# Fase 1 — Preparação e Serialização do Modelo

## Objetivo

Nesta etapa, será reaproveitado o modelo **Support Vector Machine (SVM)** desenvolvido no mini-projeto anterior de classificação de dígitos manuscritos utilizando a base de dados MNIST.

O SVM apresentou acurácia de aproximadamente **98,3% no conjunto de teste**, superando os outros modelos avaliados naquele projeto. Por esse motivo, ele foi escolhido para ser disponibilizado por meio de uma API desenvolvida com FastAPI.

## O que será realizado?

1. Carregar a base de dados MNIST.
2. Separar os dados em conjuntos de treinamento, validação e teste.
3. Normalizar os pixels das imagens para a faixa de 0 a 1.
4. Treinar o modelo SVM com os parâmetros definidos no mini-projeto.
5. Avaliar seu desempenho com métricas de classificação.
6. Serializar o modelo em um arquivo `.joblib`, permitindo que a API o carregue posteriormente sem precisar treiná-lo novamente.

## Por que serializar o modelo?

O treinamento de um algoritmo de Machine Learning pode exigir tempo e recursos computacionais. A serialização permite salvar o modelo treinado em um arquivo e reutilizá-lo em outras aplicações.

Assim, a API poderá carregar o modelo durante sua inicialização e utilizá-lo para classificar novos dígitos, sem executar novamente o processo de treinamento a cada requisição.

**Observação:** o modelo será treinado com imagens cujos pixels estejam normalizados entre 0 e 1. Por isso, os dados recebidos pela API deverão seguir o mesmo padrão de entrada.


### Divisão dos dados

Os dados foram divididos em três conjuntos, utilizando amostragem estratificada para preservar a proporção das classes de dígitos:

- **Treinamento (70%):** utilizado para ajustar o modelo SVM.
- **Validação (10%):** reservado para avaliar o modelo durante o desenvolvimento e apoiar eventuais ajustes de parâmetros.
- **Teste (20%):** utilizado para calcular as métricas finais de desempenho.

Nesta versão do projeto, o conjunto de validação foi separado, mas não foi utilizado para selecionar parâmetros do modelo. As métricas finais apresentadas foram calculadas no conjunto de teste, que permaneceu separado do treinamento.

Essa separação ajuda a avaliar o desempenho do modelo em dados que não foram utilizados diretamente para seu ajuste.

In [1]:

from pathlib import Path
import json
import time
import joblib
import numpy as np

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# 1. Criar a pasta onde o modelo será salvo
PASTA_MODELOS = Path("models")
PASTA_MODELOS.mkdir(parents=True, exist_ok=True)

# 2. Carregar a base de dados MNIST
dados = fetch_openml(
    "mnist_784",
    version=1,
    as_frame=False,
    parser="auto"
)

X = dados.data
y = dados.target.astype(int)

# Garantir que os rótulos sejam numéricos
y = y.astype(int)

print("Dimensões das imagens:", X.shape)
print("Quantidade de rótulos:", len(y))

# 3. Separar os dados em treino, validação e teste
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.125,
    random_state=42,
    stratify=y_train_val
)

# 4. Normalizar os pixels para a faixa de 0 a 1
X_train = X_train.astype(np.float32) / 255.0
X_val = X_val.astype(np.float32) / 255.0
X_test = X_test.astype(np.float32) / 255.0

print("Treinamento:", X_train.shape)
print("Validação:", X_val.shape)
print("Teste:", X_test.shape)

# 5. Criar e treinar o modelo SVM
modelo_svm = SVC(
    C=10,
    kernel="rbf"
)

inicio = time.time()
modelo_svm.fit(X_train, y_train)
tempo_treinamento = time.time() - inicio

# 6. Avaliar o modelo no conjunto de teste
y_pred = modelo_svm.predict(X_test)

metricas = {
    "modelo": "SVM",
    "acuracia": float(accuracy_score(y_test, y_pred)),
    "precisao_weighted": float(
        precision_score(y_test, y_pred, average="weighted")
    ),
    "recall_weighted": float(
        recall_score(y_test, y_pred, average="weighted")
    ),
    "f1_weighted": float(
        f1_score(y_test, y_pred, average="weighted")
    ),
    "tempo_treinamento_segundos": round(tempo_treinamento, 2)
}

print("\nMétricas do modelo:")
for nome, valor in metricas.items():
    print(f"{nome}: {valor}")

# 7. Salvar o modelo treinado
caminho_modelo = PASTA_MODELOS / "svm_mnist.joblib"
joblib.dump(modelo_svm, caminho_modelo)

# 8. Salvar as métricas em um arquivo JSON
caminho_metricas = PASTA_MODELOS / "metricas.json"

with open(caminho_metricas, "w", encoding="utf-8") as arquivo:
    json.dump(metricas, arquivo, indent=4, ensure_ascii=False)

print("\nModelo salvo em:", caminho_modelo)
print("Métricas salvas em:", caminho_metricas)


Dimensões das imagens: (70000, 784)
Quantidade de rótulos: 70000
Treinamento: (49000, 784)
Validação: (7000, 784)
Teste: (14000, 784)

Métricas do modelo:
modelo: SVM
acuracia: 0.983
precisao_weighted: 0.9830040759001173
recall_weighted: 0.983
f1_weighted: 0.9829912909280121
tempo_treinamento_segundos: 362.76

Modelo salvo em: models\svm_mnist.joblib
Métricas salvas em: models\metricas.json


## Resultados da Fase 1

O modelo SVM foi treinado com 49.000 imagens do conjunto MNIST, utilizando os parâmetros `C=10` e `kernel="rbf"`. Os dados foram divididos em treinamento (70%), validação (10%) e teste (20%), preservando a proporção das classes por meio do parâmetro `stratify`.

Após a normalização dos pixels, o modelo foi avaliado em 14.000 imagens reservadas para teste. Foram obtidos os seguintes resultados:

- **Acurácia:** 98,30%.
- **Precisão ponderada:** aproximadamente 98,30%.
- **Recall ponderado:** 98,30%.
- **F1-score ponderado:** aproximadamente 98,30%.
- **Tempo de treinamento:** 206,84 segundos.

Os resultados indicam bom desempenho na classificação de dígitos manuscritos. O tempo de treinamento também foi registrado para documentar o custo computacional da preparação do modelo.

Por fim, o modelo foi serializado no arquivo `models/svm_mnist.joblib`, enquanto as métricas foram armazenadas em `models/metricas.json`. Esses arquivos permitem reaproveitar o modelo sem precisar treiná-lo novamente durante cada inicialização da API, embora o artefato ainda precise ser carregado na memória quando o serviço iniciar.

### Teste do modelo serializado

Nesta etapa, o modelo SVM salvo em disco é carregado novamente com Joblib e utilizado para realizar uma previsão sobre uma amostra do conjunto de teste.

O objetivo é verificar se o modelo pode ser recuperado após o treinamento e continuar realizando classificações.

**Ordem de execução:** esta célula de teste depende das variáveis `X_test` e `y_test`, criadas durante a etapa de preparação dos dados. Portanto, execute primeiro a célula de treinamento e, em seguida, esta célula de teste.

O resultado apresenta o dígito previsto pelo modelo e o valor real da amostra, permitindo verificar se a classificação está correta.

In [2]:

#confirmar que o modelo salvo pode ser carregado e utilizado para prever uma imagem.

modelo_carregado = joblib.load("models/svm_mnist.joblib")

# Utiliza uma imagem do conjunto de teste
imagem_teste = X_test[0].reshape(1, -1)
previsao = modelo_carregado.predict(imagem_teste)[0]

print("Modelo carregado com sucesso!")
print("Dígito previsto:", previsao)
print("Dígito correto:", y_test[0])

Modelo carregado com sucesso!
Dígito previsto: 7
Dígito correto: 7


# Fase 2 — Estrutura modular da API FastAPI

Nesta etapa, o projeto foi organizado em módulos para separar as responsabilidades da aplicação. Essa organização facilita a manutenção, os testes e a evolução do sistema.

### Organização dos arquivos

- **`app/main.py`** — ponto de entrada da aplicação FastAPI. Configura a API e carrega o modelo SVM durante a inicialização, utilizando o mecanismo `lifespan`.
- **`app/schemas.py`** — define os formatos dos dados de entrada e saída e valida as informações recebidas com Pydantic.
- **`app/services.py`** — concentra a lógica de processamento das imagens e a utilização do modelo para realizar previsões.
- **`app/routers.py`** — define os endpoints que recebem as requisições e retornam as respostas da API.
- **`models/svm_mnist.joblib`** — armazena o modelo SVM treinado e serializado.
- **`models/metricas.json`** — registra as métricas obtidas na avaliação do modelo.
- **`notebooks/`** — organiza os notebooks utilizados durante o desenvolvimento e a experimentação.
- **`requirements.txt`** — registra as dependências e suas versões.

### Por que separar a aplicação em módulos?

A separação permite que cada arquivo tenha uma responsabilidade específica. Assim, alterações na validação dos dados, no processamento das imagens ou nas rotas podem ser realizadas sem concentrar toda a lógica em um único arquivo.

Outro aspecto importante é o carregamento do modelo: ele ocorre na inicialização da aplicação, evitando que o arquivo do modelo seja carregado novamente a cada requisição.

### Organização inicial do projeto

A estrutura de diretórios foi criada para separar o modelo treinado, os componentes da API e os notebooks de desenvolvimento.

- `app/`: contém os módulos responsáveis pela aplicação FastAPI.
- `models/`: armazena o modelo SVM serializado e o arquivo de métricas.
- `notebooks/`: organiza os notebooks utilizados no desenvolvimento.

O arquivo `app/__init__.py` identifica o diretório `app` como um pacote Python, permitindo organizar os módulos da aplicação.

Nesta versão final, a estrutura já está criada. Portanto, esta seção documenta a organização adotada, sem precisar recriar os diretórios.


### Fase 2.1 — Validação dos dados com Pydantic

O arquivo `app/schemas.py` define os formatos de entrada e saída da API utilizando Pydantic.

Na requisição de previsão, a API espera uma lista com exatamente 784 valores inteiros, correspondentes aos pixels de uma imagem de 28 × 28.

Cada pixel deve estar entre 0 e 255. A validação também impede campos adicionais não previstos no esquema.

O uso de esquemas permite rejeitar requisições inválidas antes de executar a previsão do modelo, tornando a API mais consistente e segura.


### Fase 2.2 — Dependências e execução da API

As bibliotecas necessárias para executar o projeto estão registradas no arquivo `requirements.txt`, com suas versões fixadas para facilitar a reprodução do ambiente.

**1. Instalar as dependências**

No terminal PowerShell, na pasta raiz do projeto, execute:

```powershell
python -m pip install -r requirements.txt
```

**2. Iniciar a API**

Com o ambiente virtual ativado, execute:

```powershell
python -m uvicorn app.main:app --reload
```

**3. Acessar a documentação interativa**

Depois que o servidor iniciar, abra no navegador:

http://127.0.0.1:8000/docs

Nessa página é possível consultar os endpoints e realizar testes diretamente pela interface do FastAPI.

**Função das principais bibliotecas**

- **FastAPI:** permite construir a API e definir seus endpoints.
- **Uvicorn:** executa a aplicação e recebe as requisições HTTP.
- **Pydantic:** valida os dados recebidos pela API.
- **Joblib:** carrega o modelo SVM salvo em disco.
- **Pillow:** auxilia no processamento das imagens enviadas para previsão.

# Fase 3 — Disponibilização do modelo por meio da API

Após o treinamento e a serialização do modelo SVM, a próxima etapa consiste em disponibilizar suas previsões por meio de uma API REST desenvolvida com FastAPI.

A API permite que outras aplicações enviem dados para o modelo e recebam as previsões sem precisar executar novamente o treinamento.

### Endpoints da aplicação

| Endpoint | Finalidade |
|---|---|
| `GET /health` | Verifica o estado da aplicação e informa se o modelo está carregado. |
| `POST /predict` | Recebe uma lista com 784 valores de pixels e retorna o dígito previsto. |
| `POST /predict/image` | Recebe uma imagem enviada pelo usuário, realiza o processamento necessário e retorna o dígito previsto. |

### Carregamento do modelo

O modelo treinado é carregado durante a inicialização da aplicação, utilizando o mecanismo `lifespan` do FastAPI. Dessa forma, ele fica disponível para atender às requisições sem precisar ser carregado novamente a cada previsão.

### Validação e processamento

Os dados recebidos são validados antes da previsão. No endpoint `/predict`, a requisição deve conter exatamente 784 valores inteiros, correspondentes aos pixels de uma imagem de 28 × 28.

No endpoint `/predict/image`, a aplicação recebe a imagem, prepara os pixels para o modelo e realiza a previsão. Imagens inválidas ou requisições incompatíveis podem resultar em respostas de erro apropriadas.

### Testes da API

A documentação interativa disponível em `/docs` permite testar os endpoints e verificar suas respostas.

Essa etapa demonstra como um modelo treinado em Python pode ser integrado a uma aplicação acessível por requisições HTTP.

# Conclusão

Neste projeto, foi desenvolvido um modelo de Machine Learning para reconhecimento de dígitos manuscritos utilizando a base de dados MNIST.

Após a preparação dos dados, o treinamento e a avaliação de diferentes algoritmos, o modelo SVM foi selecionado para a solução final por apresentar o melhor desempenho entre os modelos avaliados na etapa anterior.

O modelo alcançou aproximadamente **98,30% de acurácia** no conjunto de teste, com precisão ponderada, recall ponderado e F1-score ponderado também próximos de 98,30%. Esses resultados demonstram um bom desempenho na classificação de dígitos manuscritos presentes na base utilizada.

Após o treinamento, o modelo foi serializado com Joblib, permitindo seu armazenamento e carregamento posterior sem a necessidade de realizar um novo treinamento.

Na etapa seguinte, o modelo foi integrado a uma API REST desenvolvida com FastAPI. A aplicação disponibiliza endpoints para verificar seu estado, receber vetores de pixels e realizar previsões a partir de imagens enviadas pelo usuário. A organização modular, a validação dos dados com Pydantic e o carregamento do modelo na inicialização contribuem para a manutenção e a utilização da solução.

Como possibilidades de evolução, destacam-se o aprimoramento do pré-processamento de imagens manuscritas, a investigação dos erros de classificação e a ampliação dos testes com imagens externas à base MNIST.

Conclui-se que o projeto permitiu percorrer as principais etapas de uma solução de Inteligência Artificial aplicada: preparação dos dados, treinamento, avaliação, serialização do modelo e disponibilização das previsões por meio de uma API.